## Goal: - 

This is majorly the v0 code, but an improved written version of the same, you will not find similar comments and learning, but optimized architecture of training code

### Step-1: - Setting up imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
from pathlib import Path
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()
print(f"Project Root: {project_root}")
ROOT_DATASET_DIR = project_root / "datasets" 
DATASET_DIR = ROOT_DATASET_DIR /"tinyshakespeare"

Project Root: /Users/vedaangchopra/all_data/complete_technical_work/Reference and Learning Content/Deep-Learning


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

### Step-2: - Setting up HyperParams

#### Using GPU
When setting device to GPU, ensure
1. The data is in the GPU
2. The model is in the GPU
3. The context for model prediction is in the GPU

In [4]:
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000    ## Total iterations
eval_interval  = 500   ##Evaluating loss at this interval,
learning_rate = 3e-4 
eval_iters = 200
num_heads = 4
n_layer =6
dropout = 0.2
n_embed = 384
head_size= n_embed




# device = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
    print(f"Using hardware accelerator backend: {device.type}")
else:
    device = torch.device("cpu")
    print("No accelerator found. Defaulting to CPU.")


Using hardware accelerator backend: mps


In [5]:
torch.manual_seed(1337)

### Step-3: - Preparing the dataset 
Here the dataset selected the dataset is Tiny Shakespeare Dataset (All of shakespeare concantenated into one file)

In [6]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open( (DATASET_DIR /'input.txt').resolve(), 'r', encoding='utf-8') as f:
    text = f.read()

In [7]:
## First lets build the vocabulary -> That means the total number of unique characters present in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("Vocabulary Size:", vocab_size)
# This is a lookup table
stoi = { ch:i for i,ch in enumerate(chars)} # string to integer
itos = { i:ch for i,ch in enumerate(chars)} # integer to string
encode = lambda s: [stoi[c] for c in s] # encoder: take a string and output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary Size: 65


In [8]:
# Because of character level, we have long training sequences
print(encode("hi there"))
print(decode(encode("Hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
Hi there


In [9]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [10]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [36]:
print(len(data) - block_size, batch_size)
ix = torch.randint(len(data) - block_size, (batch_size,))
ix.shape

1115138 64


torch.Size([64])

#### Evaluating loss at interval

Since at each boss the loss can be jittery, thus evaluate it at an interval.

Torch.no_grad -> This is an optimization technique, for this the model doesn't maintain the gradients during the function. Here the Pytorch is efficient, since backprop is not being called.

Eval mode and Training mode -> Helps us to set behavior of model layers

In [13]:
@torch.no_grad()
def estimate_loss():
    out ={}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits , loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Step-4: - Basic Model Creation
Now that we have prepared data, let us feed it to the model

In [14]:
from typing import Any
from attention import Head, MultiHeadAttention

In [15]:
from typing import Any

class FeedForward(nn.Module):
    '''A simple linear layer followed by non-linearity'''
    
    def __init__(self, n_embed, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed, n_embed),
            nn.Dropout(dropout)
        ).to(device)
    
    def forward(self, x):
        return self.net(x)



In [16]:


class Block(nn.Module):
    """ Transformer block: communication followed by computation  
    
    Communication -> Attention
    Computation -> Feed forward Layers
    
    """
    
    
    def __init__(self, 
                n_embed,
                n_head,
                *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(num_heads=num_heads,
                                    n_embed=n_embed,
                                    block_size=block_size,
                                    head_size=head_size,
                                    dropout=dropout,
                                    device=device).to(device)
        self.ffwd = FeedForward(n_embed).to(device)
        self.ln1 = nn.LayerNorm(n_embed).to(device)
        self.ln2 = nn.LayerNorm(n_embed).to(device)
        
    def forward(self, x):
        x = x+ self.sa(self.ln1(x))
        x = x+ self.ffwd(self.ln2(x))
        return x

In [17]:

from typing import Any


class GPTLanguageModel(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) ## (B, T, C)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed=n_embed,
                                            n_head=num_heads) for _ in range(n_layer) ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self.__init_weights)
        

    def __init_weights(self,module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std =0.02)
    
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        ## idx and targets have the same weight
        tok_emd= self.token_embedding_table(idx)
        pos_emd = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emd + pos_emd
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss =None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            
            return logits, loss
        
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            ### We have to ensure that the block_size, is maintained in input, if input length is beyond block_size we need to scale it
            idx_cond = idx[:, -block_size]
            
            logits, loss = self(idx_cond)
            
            logits = logits[:, -1, :]
            
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat((idx, idx_next), dim =1)
        return idx        
            

In [18]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.788929 M parameters


In [ ]:
# from typing import Any

# class BiagramLanguageModel(nn.Module):
#     def __init__(self, vocab_size, *args: Any, **kwargs: Any) -> None:
#         super().__init__(*args, **kwargs)
        
#         ## The table here is feature embedding or word embedding table. 
#         ## Whenever we feed the batch of 2 dimension data [B,T] it does a lookup and then we get a (B,T,C(feature))
#         # Since now we are using a different embedding dimension, we need to add linear layer to get logits
#         self.token_embed = nn.Embedding(vocab_size, n_embed) ## (B, T, C)
#         ## Currently we just try to take in the tokens only, we need to have position information also
#         self.position_embedding_table = nn.Embedding(block_size, n_embed)
        
#         ## Attention Head
#         self.sa_heads = MultiHeadAttention(num_heads, 
#                                         n_embed, 
#                                         block_size,
#                                         head_size,
#                                         dropout,
#                                         device)
#         self.lm_head = nn.Linear(n_embed, vocab_size)
    
#     def forward(self,idx, targets=None):
#         B, T = idx.shape
#         token_emb = self.token_embed(idx)  # Here the input is (B, T, C)
#         pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C) ; Batch addition operation
        
#         x = token_emb + pos_emb
#         x= self.sa_heads(x)      # Apply self attention head
#         logits = self.lm_head(x)    # Here the input is (B, T, vocab_size)
        
        
#         ## Cross entropy loss except input (B,C,T) -> MultiDimensional Input the channels should be second input
#         if targets is None:
#             loss = None
#         else:
#             ## here we reshape logits
#             B,T,C = logits.shape
#             logits = logits.view(B*T, C)
#             ## We create it to 2D dimension
#             targets = targets.view(-1) # (B, T)
#             loss = F.cross_entropy(logits, targets)
#         return logits, loss
    
#     def generate(self, idx, max_new_tokens):
#         ## idx here is the input, The (B,T) given here in the current context
#         for _ in range(max_new_tokens):
#             ## We need to ensure input cannot be greater than block size, if so then we can't predict, as it will overflow
#             idx_cond = idx[:, -block_size:]
        
#             ### Get the Predictions
#             logits, loss = self(idx_cond)
            
#             ## Focus on the last Step (output of last, input of next)
#             logits = logits[:, -1, :]
            
#             ## Apply softmax for probabilities
#             probs = F.softmax(logits, dim=1) # Becomes (B,C)
            
#             idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
#             idx =torch.cat([idx, idx_next], dim=1) #(B, T+1)
#         return idx
        
# model= BiagramLanguageModel(vocab_size)
# model.to(device)

#### Step-5: -Training this Model

In [19]:
## Set the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [22]:
xb, yb = get_batch('train')

torch.Size([256])

In [ ]:
for iter in range(max_iters):
    ### Every once in a while evaluate the loss on train and val sets
    if iter% eval_interval ==0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    ## Sample a batch of data
    xb, yb = get_batch('train')
    
    ## Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

### Step-6: - Generating from Model

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))